# 06 · Coupled policy maps on a four-regime chain

## Question

What changes when the best policy for one operating regime depends on the policy chosen for another regime?

## Concept

**Informatics view.** The local score matrix supplies node scores and the regime graph supplies pairwise penalties. The graph describes relationships between engine operating conditions. The Ising Hamiltonian, introduced later in the notebook, encodes the resulting objective in quantum operators.

With only local performance, maximize $J_\mathrm{local}=\sum_i J_{i,\Pi(i)}$. Each row can be optimized independently because each choice affects just one term. Add a switching penalty across selected pairs of regimes:

$$J_\mathrm{map}=\sum_i J_{i,\Pi(i)}-\beta\sum_{(i,k)\in E} C(\Pi(i),\Pi(k)).$$

| Object | Meaning here |
|---|---|
| $\Pi(i)$ | Policy assigned to regime $i$; $i,k$ index regimes and $j,l$ index policies |
| $J_{ij}$ | Local performance of policy $j$ in regime $i$ |
| $E$ | Edges of the regime graph. Only neighboring regimes on the chain interact. |
| $C(j,l)$ | 0 for the same policy and 1 for different policies |
| $\beta$ | Score lost per policy transition across an edge |

## Worked example

Use an **illustrative** chain $\rho_1 - \rho_2 - \rho_3 - \rho_4$ with two policies. Scores are in arbitrary units, and higher is better.

| Regime | $\pi_1$ | $\pi_2$ |
|---|---:|---:|
| $\rho_1$ | 4 | 0 |
| $\rho_2$ | 1 | 4 |
| $\rho_3$ | 4 | 0 |
| $\rho_4$ | 0 | 6 |

The row winners alternate. Their map 1212 scores $18-3\beta$. The map 1112 gives up three local points at $\rho_2$ but saves two transitions, scoring $15-\beta$. Here the digits are **policy numbers in regime order**.

## Exercise / prediction

Before running the cells, predict whether 1212 or 1112 wins at $\beta=2$. At what value do their scores tie? What map would you prefer if switching were very costly?

In [1]:
from itertools import product

import numpy as np
from IPython.display import Markdown, display

J = np.array([[4, 0], [1, 4], [4, 0], [0, 6]])
edges = [(0, 1), (1, 2), (2, 3)]
C = np.array([[0, 1], [1, 0]])
beta = 2.0
n_regimes, n_policies = J.shape

maps = {}
for policy_map in product(range(n_policies), repeat=n_regimes):
    local = sum(J[i, j] for i, j in enumerate(policy_map))
    transitions = sum(int(policy_map[i] != policy_map[k]) for i, k in edges)
    switch_units = sum(C[policy_map[i], policy_map[k]] for i, k in edges)
    maps[policy_map] = (local, transitions, switch_units)

def map_label(policy_map):
    return "".join(str(j + 1) for j in policy_map)

independent = tuple(J.argmax(axis=1))
scores = {policy_map: local - beta * switch_units
          for policy_map, (local, _, switch_units) in maps.items()}
best_score = max(scores.values())
winners = {policy_map for policy_map, score in scores.items() if score == best_score}
assert len(maps) == 16
assert maps[independent][0] == max(local for local, _, _ in maps.values())
assert winners == {(0, 0, 0, 1)} and best_score == 13

lines = ["| Map | Local performance | Transitions | Coupling penalty (β=2) | Total |",
         "|---|---:|---:|---:|---:|"]
for policy_map, (local, transitions, switch_units) in maps.items():
    label = map_label(policy_map)
    if policy_map in winners:
        label = f"**{label}**"
    lines.append(f"| {label} | {local} | {transitions} | {beta * switch_units:g} | {scores[policy_map]:g} |")
display(Markdown("\n".join(lines)))

| Map | Local performance | Transitions | Coupling penalty (β=2) | Total |
|---|---:|---:|---:|---:|
| 1111 | 9 | 0 | 0 | 9 |
| **1112** | 15 | 1 | 2 | 13 |
| 1121 | 5 | 2 | 4 | 1 |
| 1122 | 11 | 1 | 2 | 9 |
| 1211 | 12 | 2 | 4 | 8 |
| 1212 | 18 | 3 | 6 | 12 |
| 1221 | 8 | 2 | 4 | 4 |
| 1222 | 14 | 1 | 2 | 12 |
| 2111 | 5 | 1 | 2 | 3 |
| 2112 | 11 | 2 | 4 | 7 |
| 2121 | 1 | 3 | 6 | -5 |
| 2122 | 7 | 2 | 4 | 3 |
| 2211 | 8 | 1 | 2 | 6 |
| 2212 | 14 | 2 | 4 | 10 |
| 2221 | 4 | 1 | 2 | 2 |
| 2222 | 10 | 0 | 0 | 10 |

## Result · vary the switching weight

At β = 0 the coupling vanishes. As β rises, the best map can give up local score to avoid transitions. The sweep reports all ties at the weights shown.

In [2]:
lines = ["| β | Optimal map(s) | Local performance | Transitions | Best objective |",
         "|---:|---|---:|---:|---:|"]
for weight in (0, 1, 1.5, 2, 4, 5, 6):
    values = {policy_map: local - weight * switch_units
              for policy_map, (local, _, switch_units) in maps.items()}
    best = max(values.values())
    for policy_map, value in values.items():
        if value == best:
            local, transitions, _ = maps[policy_map]
            lines.append(f"| {weight:g} | {map_label(policy_map)} | {local} | {transitions} | {best:g} |")
display(Markdown("\n".join(lines)))

| β | Optimal map(s) | Local performance | Transitions | Best objective |
|---:|---|---:|---:|---:|
| 0 | 1212 | 18 | 3 | 18 |
| 1 | 1212 | 18 | 3 | 15 |
| 1.5 | 1112 | 15 | 1 | 13.5 |
| 1.5 | 1212 | 18 | 3 | 13.5 |
| 2 | 1112 | 15 | 1 | 13 |
| 4 | 1112 | 15 | 1 | 11 |
| 5 | 1112 | 15 | 1 | 10 |
| 5 | 2222 | 10 | 0 | 10 |
| 6 | 2222 | 10 | 0 | 10 |

**Exercise check.** The two illustrated lines cross at $\beta=1.5$. At $\beta=2$, 1112 wins globally. At $\beta=6$, 2222 wins with zero transitions.

## Extend the QUBO

Let $q_{ij}=1$ when regime $i$ selects policy $j$. The first two terms are the performance and one-hot terms from Notebook 03. For an edge $(i,k)$, a product $q_{ij}q_{kl}$ activates only if both choices are selected. With the mismatch cost $C$, only the two cross-policy products on each edge contribute.

$$E_\mathrm{QUBO} =
-\sum_{ij}J_{ij}q_{ij}
+ A\sum_i(\sum_j q_{ij}-1)^2
+\beta\sum_{(i,k)\in E}\sum_{j,l} C(j,l)q_{ij}q_{kl}.$$

$E_\mathrm{QUBO}$ denotes the total energy. $A$ is the positive one-hot penalty, and all energy terms have the same illustrative units. This extends the binary penalty formulation [1].

Use $A=\sum_{ij}|J_{ij}|+1=20$. An infeasible state pays at least 20, can gain at most 19 from the local terms, and pays a nonnegative coupling cost. Its energy is therefore at least 1. The feasible optimum here has energy −13. Check all assignments explicitly.

In [3]:
A = int(np.abs(J).sum()) + 1
n_qubits = J.size
constant = n_regimes * A
linear = -J.ravel().astype(float) - A
quadratic = {(2*i, 2*i + 1): 2*A for i in range(n_regimes)}
for i, k in edges:
    for j, l in product(range(n_policies), repeat=2):
        weight = beta * C[j, l]
        if weight:
            quadratic[2*i + j, 2*k + l] = weight

# Basis index uses qubit 0 as the least significant bit.
labels = [f"{index:0{n_qubits}b}" for index in range(2**n_qubits)]
bits = np.array([list(map(int, label[::-1])) for label in labels])
rows = bits.reshape(-1, n_regimes, n_policies)
feasible = (rows.sum(axis=2) == 1).all(axis=1)
energies = constant + bits @ linear
for (i, k), weight in quadratic.items():
    energies += weight * bits[:, i] * bits[:, k]

direct = -(rows * J).sum(axis=(1, 2)) + A * ((rows.sum(axis=2) - 1)**2).sum(axis=1)
for i, k in edges:
    mismatch = ((rows[:, i] @ C) * rows[:, k]).sum(axis=1)
    direct = direct + beta * mismatch
np.testing.assert_array_equal(energies, direct)

decoded = {i: tuple(rows[i].argmax(axis=1)) for i in np.flatnonzero(feasible)}
assert len(decoded) == len(maps)
assert set(decoded.values()) == set(maps)
for i, policy_map in decoded.items():
    assert energies[i] == -scores[policy_map]
ground = energies == energies.min()
assert np.all(feasible[ground])
assert {decoded[i] for i in np.flatnonzero(ground)} == winners
print(f"All {len(decoded)} feasible maps agree with exact enumeration.")
print(f"Across {len(energies)} binary states, QUBO ground energy: {energies.min():g}.")

All 16 feasible maps agree with exact enumeration.
Across 256 binary states, QUBO ground energy: -13.


## Verify the Ising mapping

Use $q_i=(1-Z_i)/2$ [2], with $q_i$ the flattened selection bit, $Z_i$ its Pauli operator, $l_i$ its linear coefficient and $w_{ik}$ the coefficient of a bit pair. Each $l_iq_i$ contributes $l_i/2$ to the offset and $-l_i/2$ to the $Z_i$ field. Each $w_{ik}q_iq_k$ adds $w_{ik}/4$ to the offset, subtracts it from both fields, and adds it to $Z_iZ_k$.

In [4]:
from qiskit.quantum_info import SparsePauliOp

offset = constant + linear.sum() / 2
fields = -linear / 2
zz = {}
for (i, k), weight in quadratic.items():
    offset += weight / 4
    fields[i] -= weight / 4
    fields[k] -= weight / 4
    zz[i, k] = weight / 4

hamiltonian = SparsePauliOp.from_sparse_list(
    [("Z", [i], field) for i, field in enumerate(fields) if field]
    + [("ZZ", [i, k], weight) for (i, k), weight in zz.items()],
    num_qubits=n_qubits,
)
ising_energies = hamiltonian.to_matrix().diagonal().real
np.testing.assert_allclose(energies, ising_energies + offset, rtol=0, atol=1e-12)
print(f"All {len(labels)} QUBO and Ising energies agree after adding {offset:g}.")
print(f"Logical qubits: {n_qubits}; QUBO quadratic couplings: {len(quadratic)}; ZZ terms: {len(zz)}.")
print(f"Couplings: {n_regimes} one-hot terms and {len(zz) - n_regimes} inter-regime terms.")

All 256 QUBO and Ising energies agree after adding 33.5.
Logical qubits: 8; QUBO quadratic couplings: 10; ZZ terms: 10.
Couplings: 4 one-hot terms and 6 inter-regime terms.


## Sample with depth-one QAOA

Reuse the statevector angle search and Aer sampling pattern from Notebook 05. The mixer angle is called $\theta$ here to distinguish it from the switching weight $\beta$. A small grid minimizes expected penalized energy, then Aer samples the chosen logical circuit. All shots count toward the reported feasibility and optimality rates.

In [5]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

def circuit_at(gamma, theta):
    circuit = QuantumCircuit(n_qubits)
    circuit.h(range(n_qubits))
    for i, field in enumerate(fields):
        if field:
            circuit.rz(2 * gamma * field, i)
    for (i, k), weight in zz.items():
        circuit.rzz(2 * gamma * weight, i, k)
    circuit.rx(2 * theta, range(n_qubits))
    return circuit

def expected_energy(gamma, theta):
    distribution = Statevector.from_instruction(circuit_at(gamma, theta)).probabilities()
    return float(distribution @ energies)

gamma_grid = np.linspace(0, 2*np.pi, 32, endpoint=False)
theta_grid = np.linspace(0, np.pi, 24, endpoint=False)
mean_energy, gamma, theta = min((expected_energy(g, t), g, t)
                                for g, t in product(gamma_grid, theta_grid))
circuit = circuit_at(gamma, theta)
measured = circuit.copy()
measured.measure_all()
backend = AerSimulator()
shots, seed = 8192, 42
counts = backend.run(measured, shots=shots, seed_simulator=seed).result().get_counts()
assert sum(counts.values()) == shots
fractions = np.array([counts.get(label, 0) / shots for label in labels])
sampled = [i for i in decoded if fractions[i] > 0]
print(f"Classical optimum at β={beta:g}: {', '.join(sorted(map_label(p) for p in winners))}, score {best_score:g}.")
print(f"{backend.name}: {shots} shots, seed {seed}, grid 32×24.")
print(f"Angles γ={gamma:.6f}, θ={theta:.6f}; mean ideal penalized energy {mean_energy:.3f}.")
print(f"Feasible shots: {fractions @ feasible:.2%}; optimal shots: {fractions @ ground:.2%}.")
if sampled:
    best = min(sampled, key=lambda i: energies[i])
    print(f"Best sampled map: {map_label(decoded[best])}, objective {-energies[best]:g}.")
else:
    print("No feasible map was sampled.")
one_qubit = sum(item.operation.num_qubits == 1 for item in circuit.data)
two_qubit = sum(item.operation.num_qubits == 2 for item in circuit.data)
print(f"Logical depth: {circuit.depth()}; one-qubit gates: {one_qubit}; two-qubit gates: {two_qubit}.")

Classical optimum at β=2: 1112, score 13.
aer_simulator: 8192 shots, seed 42, grid 32×24.
Angles γ=6.086836, θ=2.748894; mean ideal penalized energy 4.415.
Feasible shots: 50.50%; optimal shots: 5.87%.
Best sampled map: 1112, objective 13.
Logical depth: 7; one-qubit gates: 24; two-qubit gates: 10.


## Takeaway

Switching costs can change the exact optimum from the row-wise choice to a smoother map. The one-hot QUBO and Ising model preserve the coupled objective. Aer sampling illustrates the encoding and can be checked against the exact answer.

The resource counts are for the logical, unmeasured circuit. A real QPU has a fixed connectivity graph and native gate set. Routing and gate decomposition can increase depth and two-qubit gate counts.

## Presentation asset

Use the four-node chain beside the local matrix, then project the weight sweep to show the switch from 1212 to 1112 to 2222.

MkDocs topic for this notebook: `docs/coupled-policy.md`.

## Next step

Notebook 07 will increase the number of regimes and policies and vary graph complexity. Track the valid-map count, qubits, couplings, logical depth, and two-qubit gates. After validating the coupled and scaling cases, move to hardware-aware transpilation and noisy or device-aware simulation before using a real QPU.

### References

[1] F. Glover, G. Kochenberger and Y. Du, [A Tutorial on Formulating and Using QUBO Models](https://arxiv.org/abs/1811.11538), 2018 preprint.

[2] E. Ising, [Beitrag zur Theorie des Ferromagnetismus](https://doi.org/10.1007/BF02980577), 1925.